### Gold Layer Benchmarking - Partitioning + ZORDER (Comparative Approach)

### Objective
This notebook implements the **comparative optimization approach** for the Gold layer using:
- Partitioning
- OPTIMIZE + ZORDER

#### Why are we creating duplicate tables?
The Gold fact tables are DLT streaming tables and do not allow:
- OPTIMIZE
- ZORDER
- Partitioning changes

Therefore, we create non-streaming duplicate tables in the Gold schema:

- `coffee.gold.fact_transactions_pz`
- `coffee.gold.fact_transaction_items_pz`

#### Partitioning Strategy
Partitioning is applied using a derived column:
- `created_month = date_trunc('month', created_at)`

This supports faster month-based filtering.

#### ZORDER Strategy
ZORDER is applied on commonly filtered/joined columns:

#### fact_transactions_pz
ZORDER BY:
- store_id
- user_id
- created_at

#### fact_transaction_items_pz
ZORDER BY:
- transaction_id
- item_id
- created_at

#### Benchmarking Approach
The same 5 business benchmark queries are executed on:
- Baseline tables (original Gold facts)
- Partition + ZORDER tables (duplicates)

Runtime from the 2nd run is recorded.


In [0]:
-- =========================================================
-- STEP 1: Drop old Partition+ZORDER benchmark tables
-- =========================================================
DROP TABLE IF EXISTS coffee.gold.fact_transactions_pz;
DROP TABLE IF EXISTS coffee.gold.fact_transaction_items_pz;


In [0]:
-- =========================================================
-- STEP 2: Create Partitioned duplicate tables (non-streaming)
-- Partition column: created_month
-- =========================================================

-- Transactions partitioned table
CREATE TABLE coffee.gold.fact_transactions_pz
USING DELTA
PARTITIONED BY (created_month)
AS
SELECT
  *,
  date_trunc('month', created_at) AS created_month
FROM coffee.gold.fact_transactions;


-- Transaction items partitioned table
CREATE TABLE coffee.gold.fact_transaction_items_pz
USING DELTA
PARTITIONED BY (created_month)
AS
SELECT
  *,
  date_trunc('month', created_at) AS created_month
FROM coffee.gold.fact_transaction_items;


In [0]:
-- =========================================================
-- STEP 3: OPTIMIZE + ZORDER
-- This physically rewrites files and improves data skipping
-- =========================================================

OPTIMIZE coffee.gold.fact_transactions_pz
ZORDER BY (store_id, user_id, created_at);

OPTIMIZE coffee.gold.fact_transaction_items_pz
ZORDER BY (transaction_id, item_id, created_at);


In [0]:
-- =========================================================
-- Q1: Monthly Sales Trend
-- =========================================================
SELECT
  date_trunc('month', created_at) AS sales_month,
  ROUND(SUM(final_amount), 2) AS total_sales
FROM coffee.gold.fact_transactions_pz
GROUP BY date_trunc('month', created_at)
ORDER BY sales_month;


In [0]:
-- =========================================================
-- Q2: Store Performance (Last 3 Months)
-- =========================================================
WITH max_dt AS (
  SELECT MAX(created_at) AS max_created_at
  FROM coffee.gold.fact_transactions_pz
)
SELECT
  store_id,
  ROUND(SUM(final_amount), 2) AS total_sales
FROM coffee.gold.fact_transactions_pz
WHERE created_at >= add_months((SELECT max_created_at FROM max_dt), -3)
GROUP BY store_id
ORDER BY total_sales DESC;

In [0]:
-- =========================================================
-- Q3: Top 10 Customers by Spend
-- =========================================================
SELECT
  item_id,
  ROUND(SUM(subtotal), 2) AS total_revenue
FROM coffee.gold.fact_transaction_items_pz
GROUP BY item_id
ORDER BY total_revenue DESC
LIMIT 10;

In [0]:
-- =========================================================
-- Q4: Top 10 Menu Items by Revenue
-- =========================================================
SELECT
  item_id,
  ROUND(SUM(subtotal), 2) AS total_revenue
FROM coffee.gold.fact_transaction_items_pz
GROUP BY item_id
ORDER BY total_revenue DESC
LIMIT 10;

In [0]:
-- =========================================================
-- Q5: Join Drilldown (Store + Item Revenue)
-- =========================================================
SELECT
  t.store_id,
  i.item_id,
  ROUND(SUM(i.subtotal), 2) AS total_revenue
FROM coffee.gold.fact_transactions_pz t
JOIN coffee.gold.fact_transaction_items_pz i
  ON t.transaction_id = i.transaction_id
GROUP BY t.store_id, i.item_id
ORDER BY total_revenue DESC;

## Benchmark Results (Baseline vs Partition + ZORDER)

| Query ID | Query Name | Baseline Runtime (Run 2) | Partition+ZORDER Runtime (Run 2) |
|---------|------------|---------------------------|----------------------------------|
| Q1 | Monthly Sales Trend | 2.23 | 1.54 |
| Q2 | Store Performance (Last 3 Months) | 2.47 |1.69  |
| Q3 | Top 10 Customers by Spend | 1.69 | 1.32 |
| Q4 | Top 10 Menu Items by Revenue | 1.59 | 1.40 |
| Q5 | Join Drilldown (Store + Item Revenue) | 6.19 | 4.72 |
